In [ ]:
import os

PACK = "/kaggle/input/datasets/avanithkanamarlapudi/cellpose-offline-pack"
MODEL_PATH = os.path.join(PACK, "models", "cpsam_v2")
WHEELS = os.path.join(PACK, "wheels")

print(os.path.exists(MODEL_PATH), os.path.exists(WHEELS))

In [ ]:
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-index", "--find-links", WHEELS,
    "cellpose", "fastremap", "fill-voids", "imagecodecs",
    "natsort", "roifile", "segment_anything", "tifffile",
])
print("ok")

In [ ]:
import json
import blosc2
import cv2
import numpy as np
import pandas as pd
import torch
from cellpose import models

print("cuda:", torch.cuda.is_available())

In [ ]:
cp_model = models.CellposeModel(
    gpu=torch.cuda.is_available(),
    pretrained_model=MODEL_PATH,
)
print("model ready")

In [ ]:
TEST_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"
samples = sorted(d.replace(".zarr", "") for d in os.listdir(TEST_DIR) if d.endswith(".zarr"))
print(samples)

In [ ]:
def load_frame(zarr_path, t, shape, dtype):
    path = os.path.join(zarr_path, "0", "c", str(t), "0", "0", "0")
    with open(path, "rb") as f:
        raw = blosc2.decompress(f.read())
    return np.frombuffer(raw, dtype=dtype).reshape(shape[1:])

In [ ]:
def read_meta(zarr_path):
    with open(os.path.join(zarr_path, "0", "zarr.json")) as f:
        meta = json.load(f)
    return tuple(meta["shape"]), np.dtype(meta["data_type"])

In [ ]:
def mip_u8(vol):
    m = vol.max(0)
    return cv2.normalize(m, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

In [ ]:
def best_z(vol, x, y):
    xi = int(np.clip(round(x), 0, vol.shape[2] - 1))
    yi = int(np.clip(round(y), 0, vol.shape[1] - 1))
    return int(np.argmax(vol[:, yi, xi]))

In [ ]:
def detect(gray):
    img = gray.astype(np.float32) / 255.0
    masks, flows, styles = cp_model.eval(img, diameter=None)
    pts = []
    for lab in range(1, int(masks.max()) + 1):
        ys, xs = np.where(masks == lab)
        if len(xs):
            pts.append([xs.mean(), ys.mean()])
    return np.asarray(pts, np.float32) if pts else np.zeros((0, 2), np.float32)

In [ ]:
def add_nodes(rows, name, t, vol, xy, start_id):
    nodes = []
    nid = start_id
    for x, y in xy:
        rows.append({
            "dataset": name, "row_type": "node", "node_id": nid,
            "t": t, "z": best_z(vol, x, y), "y": int(y), "x": int(x),
            "source_id": -1, "target_id": -1,
        })
        nodes.append((nid, float(x), float(y)))
        nid += 1
    return nid, nodes

In [ ]:
def add_edges(rows, name, prev_gray, gray, prev_nodes, curr_nodes, max_dist=20):
    if not prev_nodes or not curr_nodes:
        return
    pts0 = np.array([[x, y] for _, x, y in prev_nodes], np.float32).reshape(-1, 1, 2)
    pts1, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, pts0, None)
    curr_xy = np.array([[x, y] for _, x, y in curr_nodes], np.float32)

    for i, ok in enumerate(status.ravel()):
        if not ok:
            continue
        d = np.sqrt(((curr_xy - pts1[i].ravel()) ** 2).sum(1))
        j = int(d.argmin())
        if d[j] <= max_dist:
            rows.append({
                "dataset": name, "row_type": "edge",
                "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
                "source_id": prev_nodes[i][0], "target_id": curr_nodes[j][0],
            })

In [ ]:
def track_sample(name):
    path = os.path.join(TEST_DIR, name + ".zarr")
    shape, dtype = read_meta(path)
    rows, nid = [], 1
    prev_gray, prev_nodes = None, []

    for t in range(shape[0]):
        vol = load_frame(path, t, shape, dtype)
        gray = mip_u8(vol)
        nid, curr = add_nodes(rows, name, t, vol, detect(gray), nid)
        if prev_gray is not None:
            add_edges(rows, name, prev_gray, gray, prev_nodes, curr)
        prev_gray, prev_nodes = gray, curr

    print(name, ":", nid - 1, "nodes")
    return rows

In [ ]:
all_rows = []
for name in samples:
    all_rows += track_sample(name)
print("total rows:", len(all_rows))

In [ ]:
submission = pd.DataFrame(all_rows)
submission.index.name = "id"
submission.to_csv("submission.csv")
print("saved")
submission.head()